In [20]:
!pip install scipy
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import norm

In [3]:
print(Path.cwd())

/Users/nbs/btc-eth-dashboard/pricing


In [6]:
racine = Path.cwd().parent    
dossier_btc = racine / "data" / "raw" / "currency = BTC"

print(dossier_btc)
print("existe :", dossier_btc.exists())

/Users/nbs/btc-eth-dashboard/data/raw/currency = BTC
existe : True


In [7]:
fichiers = sorted(dossier_btc.glob("date=*/snapshot_*.parquet"))

print("nombre de fichiers :", len(fichiers))
dernier = fichiers[-1]
print(dernier)

nombre de fichiers : 1
/Users/nbs/btc-eth-dashboard/data/raw/currency = BTC/date=2026-10-03/snapshot_102352.parquet


In [8]:
df = pd.read_parquet(dernier)

print(len(df))
df.head()

1002


,snapshot_ts,currency,instrument_name,expiry,strike,option_type,bid_ask,ask_price,mark_price,mark_iv,underlying_price,open_interest,volume
0,2026-10-03 10:23:52+00:00,BTC,BTC-27NOV26-73000-P,2026-11-27 08:00:00+00:00,73000.0,P,None,0.0115,0.011014,38.86,85219.640000,63.3,0.2
1,2026-10-03 10:23:52+00:00,BTC,BTC-25DEC26-104000-C,2026-12-25 08:00:00+00:00,104000.0,C,None,0.0145,0.013975,37.73,85565.460000,173.1,0.4
2,2026-10-03 10:23:52+00:00,BTC,BTC-24SEP27-60000-C,2027-09-24 08:00:00+00:00,60000.0,C,None,0.3625,0.356911,42.24,88974.800000,0.0,0.0
3,2026-10-03 10:23:52+00:00,BTC,BTC-7OCT26-78000-C,2026-10-07 08:00:00+00:00,78000.0,C,None,0.0800,0.078197,37.60,84593.916667,0.0,0.0
4,2026-10-03 10:23:52+00:00,BTC,BTC-27NOV26-70000-C,2026-11-27 08:00:00+00:00,70000.0,C,None,0.1895,0.186099,41.05,85219.640000,3.3,0.0


In [9]:
print(df.columns.tolist())
print()
print(df.dtypes)

['snapshot_ts', 'currency', 'instrument_name', 'expiry', 'strike', 'option_type', 'bid_ask', 'ask_price', 'mark_price', 'mark_iv', 'underlying_price', 'open_interest', 'volume']

snapshot_ts         datetime64[us, UTC]
currency                            str
instrument_name                     str
expiry              datetime64[us, UTC]
strike                          float64
option_type                         str
bid_ask                          object
ask_price                       float64
mark_price                      float64
mark_iv                         float64
underlying_price                float64
open_interest                   float64
volume                          float64
dtype: object


In [10]:
secondes_par_an = 365 * 24 * 3600

df["T"] = (df["expiry"] - df["snapshot_ts"]).dt.total_seconds() / secondes_par_an

In [11]:
print(df["T"].describe())
print()
print("échéances distinctes :", df["expiry"].nunique())

count    1002.000000
mean        0.263188
std         0.313098
min         0.002466
25%         0.016165
50%         0.150411
75%         0.476439
max         0.975069
Name: T, dtype: float64

échéances distinctes : 13


In [12]:
df[df["T"] < 1/365][["instrument_name", "T"]]

,instrument_name,T
6,BTC-4OCT26-93000-P,0.002466
9,BTC-4OCT26-78000-P,0.002466
40,BTC-4OCT26-81000-C,0.002466
55,BTC-4OCT26-95000-C,0.002466
58,BTC-4OCT26-87500-P,0.002466
...,...,...
929,BTC-4OCT26-84500-P,0.002466
934,BTC-4OCT26-86500-C,0.002466
935,BTC-4OCT26-85000-C,0.002466
946,BTC-4OCT26-91000-P,0.002466


In [ ]:
def bs_price(F, k, T, sigma, is_call):
    sigma_sqrt_T = sigma*np.sqrt(T)
    d1 = (np.log(F/k) + 0.5*T*sigma**2)/(sigma_sqrt_T)
    d2 = d1 - sigma_sqrt_T
    call = F*norm.cdf(d1) - k*norm.cdf(d2)
    put = k*norm.cdf(-d2) - F*norm.cdf(-d1)
    return np.where(is_call, call, put)

In [23]:
print(bs_price(100, 100, 1, 0.2, True))    # attendu ≈ 7.966
print(bs_price(100, 100, 1, 0.2, False))   # attendu ≈ 7.966

F, k, T, s = 100, 90, 0.5, 0.3
print(bs_price(F, k, T, s, True) - bs_price(F, k, T, s, False), F - k)   # deux nombres égaux (10)

k = 120
print(bs_price(F, k, T, s, True) - bs_price(F, k, T, s, False), F - k)   # deux nombres égaux (-20)

7.965567455405804
7.965567455405804
10.0 10
-19.999999999999993 -20


In [26]:
print(df["strike"].to_numpy())

[ 73000. 104000.  60000. ...  94000.  97000.  80000.]


In [29]:
F = df["underlying_price"].to_numpy()
k = df["strike"].to_numpy()
T = df["T"].to_numpy()
sigma = df["mark_iv"].to_numpy() / 100
is_call = (df["option_type"] == "C").to_numpy()   # adapte "C" à tes valeurs

prix_modele_btc = bs_price(F, k, T, sigma, is_call) / F
ecart_F = prix_modele_btc - df["mark_price"].to_numpy()

print(pd.Series(ecart_F).describe())

count    1.002000e+03
mean    -9.302393e-07
std      8.828435e-06
min     -5.766051e-05
25%     -1.284067e-06
50%      5.530917e-09
75%      1.481159e-06
max      2.889805e-05
dtype: float64


In [31]:
mark = df["mark_price"].to_numpy()
ecart_rel = np.abs(ecart_F) / mark
print(pd.Series(ecart_rel).describe())

count    1.002000e+03
mean     4.046825e-04
std      4.078362e-03
min      1.149157e-08
25%      1.853195e-05
50%      7.490849e-05
75%      2.370534e-04
max      9.460157e-02
dtype: float64


In [32]:
df["ecart"] = ecart_F
df["ecart_rel"] = ecart_F / df["mark_price"].to_numpy()

cols = ["instrument_name", "T", "mark_price", "mark_iv", "ecart", "ecart_rel"]
print(df.loc[df["ecart_rel"].abs().nlargest(10).index, cols])

         instrument_name         T    mark_price  mark_iv         ecart  \
888   BTC-4OCT26-93000-C  0.002466  4.000000e-08    44.70  3.784063e-09   
823   BTC-4OCT26-94000-C  0.002466  2.000000e-08    47.90  1.663528e-09   
55    BTC-4OCT26-95000-C  0.002466  2.000000e-08    52.32  4.655629e-10   
186   BTC-4OCT26-92000-C  0.002466  1.000000e-07    41.56  1.583187e-09   
288   BTC-4OCT26-91000-C  0.002466  9.600000e-07    41.54  4.993353e-09   
32    BTC-5OCT26-95000-C  0.005206  1.640000e-06    45.40  5.089471e-09   
356   BTC-4OCT26-75000-P  0.002466  2.730000e-06    70.73  8.050168e-09   
1000  BTC-6OCT26-97000-C  0.007945  5.420000e-06    46.41  1.509644e-08   
425   BTC-4OCT26-74000-P  0.002466  9.500000e-07    72.89 -2.318361e-09   
411   BTC-6OCT26-93000-C  0.007945  2.715000e-05    38.01  6.060953e-08   

      ecart_rel  
888    0.094602  
823    0.083176  
55     0.023278  
186    0.015832  
288    0.005201  
32     0.003103  
356    0.002949  
1000   0.002785  
425   -0.002

In [34]:
def vega(F, k, T, sigma):
    sqrt_T = np.sqrt(T)
    d1 = (np.log(F/k) + 0.5*T*sigma**2) / (sigma*sqrt_T)
    return F * norm.pdf(d1) * sqrt_T


In [35]:
def _newton(prix_usd, F, k, T, is_call, tol=1e-8, max_iter=50):
    sigma = 0.5
    for _ in range(max_iter):
        ecart = bs_price(F, k, T, sigma, is_call) - prix_usd
        if abs(ecart) < tol:
            return sigma

        v = vega(F, k, T, sigma)
        if not np.isfinite(v) or v < 1e-12:     
            return np.nan

        sigma = sigma - ecart / v

        if not np.isfinite(sigma) or sigma < 1e-4 or sigma > 5:   
            return np.nan
    return np.nan  

from scipy.optimize import brentq

def _brent(prix_usd, F, k, T, is_call, tol=1e-8):
    f = lambda s: bs_price(F, k, T, s, is_call) - prix_usd
    try:
        return brentq(f, 1e-4, 5, xtol=tol)
    except ValueError:
        return np.nan


def implied_vol(prix_usd, F, k, T, is_call):
    if T <= 0 or prix_usd <= 0:
        return np.nan
    if is_call:
        borne_basse, borne_haute = max(F - k, 0), F
    else:
        borne_basse, borne_haute = max(k - F, 0), k
    if prix_usd <= borne_basse or prix_usd >= borne_haute:
        return np.nan

    sigma = _newton(prix_usd, F, k, T, is_call)
    if np.isnan(sigma):
        sigma = _brent(prix_usd, F, k, T, is_call)
    return sigma

In [36]:
F_arr = df["underlying_price"].to_numpy()
k_arr = df["strike"].to_numpy()
T_arr = df["T"].to_numpy()
is_call_arr = (df["option_type"] == "C").to_numpy()   # adapte "C" à tes valeurs
prix_usd_arr = (df["mark_price"] * df["underlying_price"]).to_numpy()

iv = np.empty(len(df))
for i in range(len(df)):
    iv[i] = implied_vol(prix_usd_arr[i], F_arr[i], k_arr[i], T_arr[i], is_call_arr[i])

df["iv_calc"] = iv
df["iv_ref"] = df["mark_iv"] / 100
df["iv_diff"] = df["iv_calc"] - df["iv_ref"]

In [38]:
print(df["iv_calc"], df["iv_ref"], df["iv_diff"])

0       0.388566
1       0.377327
2       0.422447
3       0.376007
4       0.410554
          ...   
997     0.386374
998     0.386555
999     0.412364
1000    0.464005
1001    0.334579
Name: iv_calc, Length: 1002, dtype: float64 0       0.3886
1       0.3773
2       0.4224
3       0.3760
4       0.4105
         ...  
997     0.3864
998     0.3866
999     0.4002
1000    0.4641
1001    0.3342
Name: iv_ref, Length: 1002, dtype: float64 0      -0.000034
1       0.000027
2       0.000047
3       0.000007
4       0.000054
          ...   
997    -0.000026
998    -0.000045
999     0.012164
1000   -0.000095
1001    0.000379
Name: iv_diff, Length: 1002, dtype: float64


In [41]:
print("NaN :", df["iv_calc"].isna().sum(), "sur", len(df))

NaN : 1 sur 1002
